## Using Method A

We have two equivalent modeling choices:

### Option 1 — Treat as standard frame element (MT = 0) 

This is what you will do now for element 3.

- Use the full 6×6 local stiffness matrix (no releases).  
- $u_x = 0$ and $u_y = 0$, restrained DOFs at the starting node.  
- Leave the rotational DOF free.  

### Option 2 — Model the pin as a member release (MT = 1) 

This is what I showed in lecture for element 3.

- A pin support restrains translations but **cannot transmit moment**.  
- From an element standpoint, this is equivalent to a hinge at the start node, $b$.  
- Therefore member 3 can be modeled as **MT = 1**.

### Implement Option 1 below and demonstrate that it gives the same results as what we did in lecture for Method A

In [ ]:
import numpy as np

from helper_funcs import (
    assemble_global_stiffness_and_fef,
    partition_system,
    assemble_global_displacements,
    assemble_global_forces
)
from helper_funcs import (
    print_element,
    print_dsm_results,
    print_matrix_scaled
)


In [19]:
# -------------------------
# Problem data (hard-coded)
# -------------------------
E = 200e6          # kPa
A = 0.0065         # m^2
I = 150e-6         # m^4
L = 5.0            # m

In [20]:
# -------------------------
# Member 1 (MT=2)
# -------------------------
k1 = np.array([
    [260000.,     0.,     0., -260000.,     0.,     0.],
    [    0.,   720.,  3600.,      0.,  -720.,     0.],
    [    0.,  3600., 18000.,      0., -3600.,     0.],
    [-260000.,    0.,     0.,  260000.,    0.,     0.],
    [    0.,  -720., -3600.,     0.,   720.,     0.],
    [    0.,     0.,     0.,      0.,     0.,     0.]
], dtype=float)

# Local fixed-end vector for member 1 (given by the example result)
Qf1 = np.array([0., 60., 60., 0., 36., 0.], dtype=float)

# Transformation matrix for theta=90deg -> c=0, s=1
T1 = np.array([
    [ 0.,  1., 0., 0., 0., 0.],
    [-1.,  0., 0., 0., 0., 0.],
    [ 0.,  0., 1., 0., 0., 0.],
    [ 0.,  0., 0., 0., 1., 0.],
    [ 0.,  0., 0.,-1., 0., 0.],
    [ 0.,  0., 0., 0., 0., 1.]
], dtype=float)

# Mapping
m1 = np.array([1,2,3,4,5,6])

In [21]:

# -------------------------
# Member 2 (MT=1)
# -------------------------
k2 = np.array([
    [260000.,     0.,     0., -260000.,     0.,     0.],
    [    0.,   720.,     0.,      0.,  -720.,  3600.],
    [    0.,     0.,     0.,      0.,     0.,     0.],
    [-260000.,    0.,     0.,  260000.,     0.,     0.],
    [    0.,  -720.,     0.,      0.,   720., -3600.],
    [    0.,  3600.,     0.,      0., -3600., 18000.]
], dtype=float)

# Local fixed-end vector for member 2 (example result)
Qf2 = np.array([0., 93.75, 0., 0., 206.25, -281.25], dtype=float)

# Horizontal member -> no transformation
T2 = np.eye(6, dtype=float)

# Mapping
m2 = np.array([4,5,6,7,8,9])


In [22]:
# -------------------------
# Member 2 (MT=0)
# -------------------------

k3 = #TODO

Qf3 = np.zeros(6, dtype=float)
T3 = T1.copy()
m3 = np.array([10,11,12,7,8,9])

SyntaxError: invalid syntax (1629419328.py, line 7)

In [ ]:
k_list = [k1, k2, k3]
T_list = [T1, T2, T3]
Qf_list = [Qf1, Qf2, Qf3]
map_list = [m1, m2, m3]   # 1-based DOF maps

# Global system size (still using 1-based maps here)
ndof = int(np.max(np.concatenate(map_list)))

In [ ]:
## USER SPECIFIED

# Applied Load
F_global = np.zeros(ndof, dtype=float)
F_global[4-1] = 90.75

# Restrained DOFs
dof_restrained_1based = np.array([1, 2, 3, 10, 11], dtype=int)

# Fictitious restrained DOFs
dof_fictitious_1based = #TODO

# Combine and sort
dof_restrained_1based = np.sort(
    np.concatenate((dof_restrained_1based, dof_fictitious_1based))
)

In [ ]:
K_global, F_fef_global = assemble_global_stiffness_and_fef(
    ndof,
    k_list,
    T_list,
    Qf_list,
    map_list
    )

In [ ]:
(
    K_ff,
    K_fr,
    K_rf,
    K_rr,
    f_f,
    f_r,
    f_fef_f,
    f_fef_r,
    free_dofs,
    restrained_dofs,
) = partition_system(
    K_global,
    F_global,
    F_fef_global,
    dof_restrained_1based
)


In [ ]:
rhs = f_f - f_fef_f
u_f = np.linalg.solve(K_ff, rhs)

F_r = K_rf @ u_f + f_fef_r

u_global = assemble_global_displacements(u_f, free_dofs, restrained_dofs)
f_global_complete = assemble_global_forces(f_f, F_r, free_dofs, restrained_dofs)


In [ ]:
print_dsm_results(
    u_global,
    f_global_complete,
    dof_restrained_1based,
    dof_fictitious_1based,
    disp_in_mm=True
)

In [ ]:
# Element 1,2,3
print_element(1, u_global, m1, T1, k1, Qf1, disp_in_mm=True, dec=1)
print_element(2, u_global, m2, T2, k2, Qf2, disp_in_mm=True, dec=1)
print_element(3, u_global, m3, T3, k3, Qf3, disp_in_mm=True, dec=1)